In [4]:
# Parameters
AUDIT_TABLE = "monitoring.cfg_silver_export_load"
TIME_PARSER_POLICY = "CORRECTED"

NOTEBOOK_TIMEOUT_SECONDS = 1800


StatementMeta(, 70f501ec-6d87-4a39-a888-2b0fd5870a98, 6, Finished, Available, Finished, False)

In [2]:
import re
from bisect import bisect_right
import uuid
from collections import defaultdict
from datetime import datetime
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType, IntegerType, LongType, StringType, StructField, StructType, TimestampType
)
from pyspark.sql.window import Window

RUN_ID = str(uuid.uuid4())
STARTED_AT = datetime.utcnow()
spark.conf.set("spark.sql.legacy.timeParserPolicy", TIME_PARSER_POLICY)


def qident(value):
    return "`" + str(value).replace("`", "``") + "`"


StatementMeta(, 70f501ec-6d87-4a39-a888-2b0fd5870a98, 4, Finished, Available, Finished, False)

In [3]:
spark.sql("CREATE SCHEMA IF NOT EXISTS monitoring")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
  source_kind STRING, source_schema STRING, source_table STRING,
  target_table STRING, export_date TIMESTAMP, status STRING,
  reload BOOLEAN, attempt_count INT, run_id STRING,
  rows_read BIGINT, rows_written BIGINT, duplicate_key_count BIGINT,
  started_at TIMESTAMP, ended_at TIMESTAMP, error_message STRING,
  last_updated_at TIMESTAMP
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_pipeline_run (
  run_id STRING, pipeline_name STRING, layer STRING, source_kind STRING,
  started_at TIMESTAMP, ended_at TIMESTAMP, status STRING,
  tables_succeeded INT, tables_failed INT, rows_read BIGINT, rows_written BIGINT,
  error_message STRING
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_table_load_metric (
  run_id STRING, layer STRING, source_kind STRING, source_object STRING,
  target_object STRING, rows_read BIGINT, rows_written BIGINT,
  duplicate_key_count BIGINT, null_primary_key_count BIGINT,
  recorded_at TIMESTAMP
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_schema_drift_definition (
  table_name STRING, ordinal_position STRING, column_name STRING,
  data_type STRING, is_nullable STRING, column_default STRING,
  primary_key_name STRING, is_primary_key STRING, foreign_key_name STRING,
  referenced_schema STRING, referenced_table STRING, referenced_column STRING,
  column_description STRING, table_description STRING,
  definition_hash STRING, definition_loaded_at TIMESTAMP
) USING DELTA
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_schema_drift_event (
  run_id STRING, source_kind STRING, source_table STRING, target_table STRING,
  drift_type STRING, column_name STRING, expected_type STRING,
  actual_type STRING, referenced_table STRING, referenced_column STRING,
  drift_key STRING, status STRING, occurrence_count BIGINT,
  first_detected_at TIMESTAMP, last_detected_at TIMESTAMP,
  resolved_at TIMESTAMP, detected_at TIMESTAMP
) USING DELTA
""")
drift_event_columns = {field.name.lower() for field in spark.table("monitoring.cfg_schema_drift_event").schema.fields}
drift_event_definitions = [
    "actual_type STRING", "referenced_table STRING", "referenced_column STRING",
    "drift_key STRING", "status STRING", "occurrence_count BIGINT",
    "first_detected_at TIMESTAMP", "last_detected_at TIMESTAMP", "resolved_at TIMESTAMP",
]
missing_drift_columns = [definition for definition in drift_event_definitions if definition.split()[0].lower() not in drift_event_columns]
if missing_drift_columns:
    spark.sql(f"ALTER TABLE monitoring.cfg_schema_drift_event ADD COLUMNS ({', '.join(missing_drift_columns)})")
spark.sql("""
CREATE TABLE IF NOT EXISTS monitoring.cfg_month_end_gold_run (
  snapshot_date DATE, status STRING, reload BOOLEAN, attempt_count INT,
  run_id STRING, started_at TIMESTAMP, ended_at TIMESTAMP,
  dq_result STRING, gold_result STRING, error_message STRING,
  last_updated_at TIMESTAMP
) USING DELTA
""")


spark.sql("CREATE SCHEMA IF NOT EXISTS monitoring")
spark.sql('''
CREATE TABLE IF NOT EXISTS monitoring.cfg_archive_zip_load (
  zip_path STRING, export_date TIMESTAMP, extract_path STRING, status STRING,
  reload BOOLEAN, attempt_count INT, file_count INT, run_id STRING,
  started_at TIMESTAMP, ended_at TIMESTAMP, error_message STRING,
  first_loaded_at TIMESTAMP, last_updated_at TIMESTAMP
) USING DELTA
''')
spark.sql('''
CREATE TABLE IF NOT EXISTS monitoring.cfg_archive_file_load (
  file_path STRING, filename STRING, export_date TIMESTAMP, source_zip STRING,
  target_object STRING, status STRING, reload BOOLEAN, attempt_count INT,
  rows_read BIGINT, rows_written BIGINT, run_id STRING, started_at TIMESTAMP,
  ended_at TIMESTAMP, error_message STRING, first_loaded_at TIMESTAMP,
  last_updated_at TIMESTAMP
) USING DELTA
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS monitoring.cfg_archive_table_export_load (
  source_schema STRING, source_table STRING, export_date TIMESTAMP,
  status STRING, reload BOOLEAN, row_count BIGINT, run_id STRING,
  first_seen_at TIMESTAMP, last_updated_at TIMESTAMP, error_message STRING
) USING DELTA
''')


StatementMeta(, 70f501ec-6d87-4a39-a888-2b0fd5870a98, 5, Finished, Available, Finished, False)

DataFrame[]